# **Palmer Penguinsデータセットを用いた機械学習入門**

## 概要

かわいいペンギンのデータセットを使って、機械学習を一通り実行してみましょう。

> **教師あり学習**
> 
> 1. 一般化線形モデル (GLM) — *ロジスティック回帰による性別の予測*
> 2. ランダムフォレスト — *ペンギン3種の分類と特徴量重要度の可視化*
> 
> **教師なし学習(クラスタリング)**
> 
> 3. 階層的クラスタリング — *デンドログラムの構築と関係性を可視化*
> 4. k-means クラスタリング — *もう一つの代表的なクラスタリング法*
> 5. グラフクラスタリング — *k-NN グラフ + Louvain 法によるコミュニティ検出*
> 
> **次元圧縮による可視化**
> 
> 6. PCA / MDS (PCoA) / t-SNE / UMAP — *4手法で2次元に圧縮して比較*

## データセットについて

南極のPalmer諸島に生息する3種類のペンギン（Adelie, Chinstrap, Gentoo）の形態学的データセット。約340個体分が記録されています。

<img src="https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/man/figures/lter_penguins.png" width="600" alt="Palmer Penguins: Adelie, Chinstrap, Gentoo">

例えば、くちばし(bill)の**長さ**と**深さ**は下図のように計測されています。

<img src="https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/man/figures/culmen_depth.png" width="450" alt="Culmen length and depth illustration">

### 変数一覧

- `species`: 種(Adelie, Chinstrap, Gentoo)
- `island`: 生息する島(Biscoe, Dream, Torgersen)
- `bill_length_mm` / `bill_depth_mm`: くちばしの長さ / 深さ(mm)
- `flipper_length_mm`: ヒレの長さ(mm)
- `body_mass_g`: 体重(g)
- `sex`: 性別

### Citation

- Horst AM, Hill AP, Gorman KB (2020). **palmerpenguins: Palmer Archipelago (Antarctica) penguin data.** R package version 0.1.0. DOI: [10.5281/zenodo.3960218](https://doi.org/10.5281/zenodo.3960218). <https://allisonhorst.github.io/palmerpenguins/>
- Gorman KB, Williams TD, Fraser WR (2014). **Ecological Sexual Dimorphism and Environmental Variability within a Community of Antarctic Penguins (Genus *Pygoscelis*).** *PLoS ONE* 9(3): e90081. DOI: [10.1371/journal.pone.0090081](https://doi.org/10.1371/journal.pone.0090081)

### ライセンス
- **データ**: CC0(パブリックドメイン、Palmer Station LTER データポリシーに準拠)
- **アートワーク**: Artwork by Allison Horst (https://github.com/allisonhorst), CC-BY 4.0

## 0. 必要ライブラリのインストールと設定

In [27]:
# 初回のみ実行してください(`%pip` はカーネルと同じ Python にインストールします)
%pip install palmerpenguins scikit-learn seaborn matplotlib pandas scipy statsmodels umap-learn networkx matplotlib-fontja ptitprince

Note: you may need to restart the kernel to use updated packages.


In [28]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ---- 日本語フォントの設定(文字化け対策) ----
# 順序が重要:seaborn のテーマを「先に」設定すると rcParams を初期化するので、
# その「あと」で matplotlib-fontja を import すれば font.family が上書きされない。
sns.set_style("whitegrid")
print("kernel python :", sys.executable)
try:
    import matplotlib_fontja  # noqa: F401
    # 念のため font.sans-serif の先頭にも IPAexGothic を入れておく
    # (将来 sns.set_theme() などで font.family が 'sans-serif' に戻されても CJK が出るように)
    _sans = [f for f in plt.rcParams["font.sans-serif"] if f != "IPAexGothic"]
    plt.rcParams["font.sans-serif"] = ["IPAexGothic"] + _sans
    print("matplotlib-fontja: OK")
except ImportError as e:
    print(f"matplotlib-fontja import 失敗: {e}")
    print("→ 上のセル(%pip install ...)を先に実行してください。")
    from matplotlib import font_manager
    _candidates = [
        "IPAexGothic", "IPAGothic", "Noto Sans CJK JP", "Noto Sans JP",
        "TakaoGothic", "VL Gothic", "Hiragino Sans", "Yu Gothic", "Meiryo",
    ]
    _available = {f.name for f in font_manager.fontManager.ttflist}
    _picked = next((n for n in _candidates if n in _available), None)
    if _picked:
        plt.rcParams["font.family"] = _picked
        plt.rcParams["font.sans-serif"] = [_picked] + plt.rcParams["font.sans-serif"]
        print(f"フォールバックで {_picked!r} を使用します。")
    else:
        print("利用可能な CJK フォントが見つかりません。")
plt.rcParams["axes.unicode_minus"] = False
print("font.family    :", plt.rcParams["font.family"])
print("font.sans-serif:", plt.rcParams["font.sans-serif"][:3], "...")

kernel python : /home/qm/claude/lecture/ml/phylo_work/.venv/bin/python
matplotlib-fontja: OK
font.family    : ['sans-serif']
font.sans-serif: ['IPAexGothic', 'Arial', 'DejaVu Sans'] ...


## 1. データの取得と確認

早速データをダウンロードします。ダウンロードしたら、データの中身について必ず目で見て確認しましょう。まず大事なポイントは、
> 1. **データのクオリティ** — 取得したデータは壊れてないか。きちんと整備されたデータか。欠損値はないか。正規化はされているか。
> 1. **データのメタ情報** — どのような種類のデータか。どのように取得されたデータか。データのサイズはどれくらいか。使用する条件は満たしているか（著作権等）。
> 1. **データの分布** — 正規分布などを当てはめられそうか。分布の形状はデータ自体の特徴か、それとも人為的な原因によるものか

の３点です。

### ダウンロード

In [29]:
from palmerpenguins import load_penguins

# データセットの読み込み
df = load_penguins()

# 基本情報の確認
print("データの形状:", df.shape)
df.head()

データの形状: (344, 8)


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007


### 各列の型と欠損数

In [30]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    str    
 1   island             344 non-null    str    
 2   bill_length_mm     342 non-null    float64
 3   bill_depth_mm      342 non-null    float64
 4   flipper_length_mm  342 non-null    float64
 5   body_mass_g        342 non-null    float64
 6   sex                333 non-null    str    
 7   year               344 non-null    int64  
dtypes: float64(4), int64(1), str(3)
memory usage: 21.6 KB


### 基本統計量

In [ ]:
df.describe()

### 種ごとのサンプル数

In [ ]:
print(df["species"].value_counts())

## 2. 前処理(欠損値の除去)

ここではシンプルに欠損値を含む行はそのまま削除します。意欲のある方は、平均値補完や多重代入といったより高度な手法に置き換えてみてください。

In [ ]:
df_clean = df.dropna().reset_index(drop=True)
print(f"欠損値除去前のサンプル数: {len(df)}")
print(f"欠損値除去後のサンプル数: {len(df_clean)}")

## 3. 探索的データ解析(EDA)

まずはデータの全体像をつかみます。

### Rain Cloud Plot

`describe()` の表は要約値しか出ませんが、**Rain Cloud Plot**は同じ情報を「分布の形」「箱ひげの要約」「生の点」の3層で同時に可視化します。

- **Cloud(半バイオリン)**: カーネル密度推定による分布の形
- **Box(箱ひげ)**: 中央値・四分位範囲(IQR)・外れ値
- **Rain(ストリッププロット)**: 個々のサンプルの「雨粒」

種で色分けすると、特徴量ごとに「種をどれだけ分離できそうか」が直観的に分かります。

In [ ]:
import warnings
import ptitprince as pt

numeric_features = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
# ptitprince 内部の sns.boxplot 呼び出しが古い API のため FutureWarning が出る。
# この呼び出し中だけ無視
with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore", category=FutureWarning, module="ptitprince.*"
    )
    for ax, feat in zip(axes.flat, numeric_features):
        pt.RainCloud(
            x="species", y=feat, data=df.dropna(subset=[feat]),
            palette="Set2", bw=.2, width_viol=.6, ax=ax, orient="v",
            point_size=3, alpha=.7,
        )
        ax.set_title(feat)
        ax.set_xlabel("")

plt.suptitle("基本統計量の Rain Cloud Plot(種ごと)", y=1.00, fontsize=14)
plt.tight_layout()
plt.show()

### 散布図行列

In [ ]:
sns.pairplot(
    df_clean,
    hue="species",
    vars=["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"],
    palette="Set2",
    diag_kind="kde",
)
plt.suptitle("ペンギン3種の特徴量分布", y=1.02)
plt.show()

> **Q1：どの特徴量を見れば３種を分離できるのか予想してみよう**

## 4. 一般化線形モデル（GLM）で性別を予測

GLMは**係数の解釈ができる**のが最大の魅力です。ここでは、ロジスティック回帰に基づいて性別予測を試みます。結果を評価するとともに、p値や信頼区間もじっくりみてみましょう。

### データの準備

In [ ]:
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

# 説明変数と目的変数
features = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
X = df_clean[features].copy()
y = (df_clean["sex"] == "male").astype(int)  # male=1, female=0

# 標準化(係数の比較がしやすくなる)
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=features)

# 訓練・テスト分割
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42, stratify=y
)

### モデルの構築

In [ ]:
# statsmodels で GLM (二項分布 + ロジットリンク)
X_train_sm = sm.add_constant(X_train)
glm = sm.GLM(y_train, X_train_sm, family=sm.families.Binomial())
result = glm.fit()
print(result.summary())

### モデルの検証

In [ ]:
# テストデータでの予測精度
X_test_sm = sm.add_constant(X_test)
y_pred = (result.predict(X_test_sm) > 0.5).astype(int)

print("混同行列:")
print(confusion_matrix(y_test, y_pred))
print("\n分類レポート:")
print(classification_report(y_test, y_pred, target_names=["female", "male"]))

### 係数の可視化

In [ ]:
# 係数の描画(標準化済みなので、そのまま大小比較ができる)
coefs = result.params.drop("const").sort_values()
plt.figure(figsize=(8, 4))
coefs.plot(kind="barh", color="coral")
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("係数(標準化後)")
plt.title("ロジスティック回帰の係数(正→オス寄り、負→メス寄り)")
plt.tight_layout()
plt.show()

> **Q2：係数の大きさと符号は生物学的に考えて妥解だろうか？**

## 5. ランダムフォレストで種を分類

GLMとは異なり、複数の決定木によるアンサンブルでは非線形な関係も捉えられます。**特徴量重要度**もみてみましょう。

### モデルの構築

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 種を予測するタスクに切り替え
X = df_clean[features]
y = df_clean["species"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# ランダムフォレスト
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print(f"テスト精度: {accuracy_score(y_test, y_pred):.3f}")
print("\n分類レポート:")
print(classification_report(y_test, y_pred))

### 特徴量重要度の可視化

In [ ]:
importance = pd.Series(rf.feature_importances_, index=features).sort_values()

plt.figure(figsize=(8, 4))
importance.plot(kind="barh", color="steelblue")
plt.xlabel("Feature Importance")
plt.title("ランダムフォレストによる特徴量重要度")
plt.tight_layout()
plt.show()

### 混同行列の可視化

In [ ]:
# 混同行列のヒートマップ
cm = confusion_matrix(y_test, y_pred, labels=rf.classes_)
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=rf.classes_, yticklabels=rf.classes_
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("混同行列(ランダムフォレスト)")
plt.tight_layout()
plt.show()

> **Q3：ランダムフォレストの重要度とGLMの係数を比較してみよう。また、EDAで見えた「種を分ける」変数と一致していただろうか？**

## 6. 階層的クラスタリング（教師なし学習）

種ラベルを**一切使わず**に、特徴量だけからグループ分けができるか試してみよう。

### クラスタリングの実行

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

# 標準化(距離計算ではスケールの影響を排除)
X = df_clean[features]
X_scaled = StandardScaler().fit_transform(X)

# Ward 法でリンケージを計算
Z = linkage(X_scaled, method="ward")

# デンドログラム
plt.figure(figsize=(12, 5))
dendrogram(Z, truncate_mode="lastp", p=30, leaf_rotation=90, leaf_font_size=10)
plt.title("階層的クラスタリングのデンドログラム (Ward法)")
plt.xlabel("サンプル(またはクラスタ)")
plt.ylabel("距離")
plt.axhline(y=15, color="red", linestyle="--", label="3クラスタで切る位置")
plt.legend()
plt.tight_layout()
plt.show()

### クラスタリング結果と種ラベルの比較

In [ ]:
# 3クラスタに切り分け
clusters = fcluster(Z, t=3, criterion="maxclust")
df_clean = df_clean.copy()
df_clean["cluster"] = clusters

# 答え合わせ:本当の種ラベルとの対応をクロス集計
print("クラスタ vs 種(答え合わせ):")
print(pd.crosstab(df_clean["cluster"], df_clean["species"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.scatterplot(
    data=df_clean, x="flipper_length_mm", y="body_mass_g",
    hue="cluster", palette="Set1", ax=axes[0]
)
axes[0].set_title("クラスタリング結果(ラベル未使用)")

sns.scatterplot(
    data=df_clean, x="flipper_length_mm", y="body_mass_g",
    hue="species", palette="Set2", ax=axes[1]
)
axes[1].set_title("本当の種ラベル(答え)")

plt.tight_layout()
plt.show()

> **Q4：クラスタリングのミスをポジティブに解釈することはできるか？**

## 7. *k*-meansクラスタリング（教師なし学習）

**重心**（クラスタの中心）を反復的に更新してデータを`k`個に分けます。階層的クラスタリングと違い、事前にクラスタ数`k`を決める必要があります。ここでは**エルボー法**でクラスタ数を検討し、`k=3`で実行して種ラベルとの一致を見ます。

### クラスタ数の検討

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# 標準化
X = df_clean[features]
X_scaled = StandardScaler().fit_transform(X)

# エルボー法でクラスタ数を検討
inertias = []
ks = range(1, 11)
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(ks, inertias, marker="o")
plt.axvline(x=3, color="red", linestyle="--", alpha=0.7, label="k=3(種数)")
plt.xlabel("クラスタ数 k")
plt.ylabel("Inertia (クラスタ内平方和)")
plt.title("エルボー法によるクラスタ数の検討")
plt.legend()
plt.tight_layout()
plt.show()


### モデルの構築

In [ ]:
# k=3 で k-means を実行
kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
df_clean["kmeans_cluster"] = kmeans.fit_predict(X_scaled)

print("k-means クラスタ vs 種(答え合わせ):")
print(pd.crosstab(df_clean["kmeans_cluster"], df_clean["species"]))

# クラスタリングと真ラベルの一致度を定量評価
ari = adjusted_rand_score(df_clean["species"], df_clean["kmeans_cluster"])
nmi = normalized_mutual_info_score(df_clean["species"], df_clean["kmeans_cluster"])
print(f"\nAdjusted Rand Index (ARI): {ari:.3f}  (1.0=完全一致, 0.0=ランダム)")
print(f"Normalized Mutual Information (NMI): {nmi:.3f}")


### 結果の可視化

In [ ]:
# k-means の結果を散布図で確認(重心も表示)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.scatterplot(
    data=df_clean, x="flipper_length_mm", y="body_mass_g",
    hue="kmeans_cluster", palette="Set1", ax=axes[0]
)
axes[0].set_title("k-means クラスタリング結果(k=3)")

sns.scatterplot(
    data=df_clean, x="flipper_length_mm", y="body_mass_g",
    hue="species", palette="Set2", ax=axes[1]
)
axes[1].set_title("本当の種ラベル(答え)")

plt.tight_layout()
plt.show()


## 8. グラフクラスタリング（コミュニティ検出）

データ点を**ノード**、近傍関係を**エッジ**とするグラフを作り、そのグラフの構造から自然な分割を見つけます。

1. 各サンプルからk個の最近傍点へのエッジを張って **k-NNグラフ**を構築
2. **Louvain 法**（モジュラリティ最大化）でコミュニティを検出

クラスタ数を事前に指定しない・密度に基づく・任意形状のクラスタを捉えやすい、という *k*-means とは異なる性質を持ちます。

### グラフの構築とコミュニティ検出

In [ ]:
import networkx as nx
from sklearn.neighbors import kneighbors_graph

# k-NN グラフを構築(k=10)
k = 10
A = kneighbors_graph(X_scaled, n_neighbors=k, mode="connectivity", include_self=False)
A = 0.5 * (A + A.T)  # 対称化(無向グラフ化)

G = nx.from_scipy_sparse_array(A)
print(f"ノード数: {G.number_of_nodes()}, エッジ数: {G.number_of_edges()}")

# Louvain 法によるコミュニティ検出
communities = nx.community.louvain_communities(G, seed=42)
print(f"検出されたコミュニティ数: {len(communities)}")

graph_clusters = np.zeros(len(df_clean), dtype=int)
for i, comm in enumerate(communities):
    for node in comm:
        graph_clusters[node] = i
df_clean["graph_cluster"] = graph_clusters

print("\nグラフクラスタ vs 種(答え合わせ):")
print(pd.crosstab(df_clean["graph_cluster"], df_clean["species"]))

ari_g = adjusted_rand_score(df_clean["species"], df_clean["graph_cluster"])
nmi_g = normalized_mutual_info_score(df_clean["species"], df_clean["graph_cluster"])
print(f"\nAdjusted Rand Index (ARI): {ari_g:.3f}")
print(f"Normalized Mutual Information (NMI): {nmi_g:.3f}")


### 結果の可視化

In [ ]:
# k-NN グラフを 2 次元に配置してコミュニティを可視化
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 左:グラフのレイアウト + コミュニティで色分け
pos = nx.spring_layout(G, seed=42, k=0.15)
colors = plt.cm.Set1(np.linspace(0, 1, len(communities)))
for i, comm in enumerate(communities):
    nx.draw_networkx_nodes(G, pos, nodelist=list(comm), node_size=40,
                           node_color=[colors[i]], ax=axes[0], label=f"comm {i}")
nx.draw_networkx_edges(G, pos, alpha=0.15, ax=axes[0])
axes[0].set_title(f"k-NN グラフ + Louvain 法 (k={k}, {len(communities)} コミュニティ)")
axes[0].axis("off")
axes[0].legend(loc="best", fontsize=8)

# 右:元の特徴空間でコミュニティを色分け
sns.scatterplot(
    data=df_clean, x="flipper_length_mm", y="body_mass_g",
    hue="graph_cluster", palette="Set1", ax=axes[1]
)
axes[1].set_title("グラフクラスタリング結果(特徴空間)")

plt.tight_layout()
plt.show()


> **Q5：階層的クラスタリング・*k*-means・グラフクラスタリングの結果を比較しながら、それぞれのPros/Consを議論してみよう**

## 9. 次元圧縮による可視化(PCA / MDS / t-SNE / UMAP)

4次元の特徴量(くちばし長・深さ、ヒレ長、体重)を2次元に圧縮して散布図にします。
代表的な4手法を比較すると、それぞれの**目的関数**（最大化 or 最小化したい目標）の違いがよく分かります。

| 手法 | 種類 | 目的関数 |
| --- | --- | --- |
| **PCA** | 線形 | **分散**を最大化 |
| **MDS (PCoA)** | 線形〜非線形 | 圧縮前後での**点間距離の差**を最小化|
| **t-SNE** | 非線形 | **局所**の近傍関係(類似点同士はより近く)の最適化 |
| **UMAP** | 非線形 | **大域構造**(大域構造の誤差は最小化) は維持しつつ、局所の近傍関係を最適化|


### 次元圧縮の実行

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import MDS, TSNE

# 共通の入力(標準化済み特徴量)
X = df_clean[features]
X_scaled = StandardScaler().fit_transform(X)
species_labels = df_clean["species"].values

# PCA(主成分分析)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print(f"PCA 寄与率: PC1={pca.explained_variance_ratio_[0]:.2%}, "
      f"PC2={pca.explained_variance_ratio_[1]:.2%}, "
      f"累積={pca.explained_variance_ratio_.sum():.2%}")

# MDS (Classical MDS = PCoA 相当、ユークリッド距離)
mds = MDS(n_components=2, metric="euclidean", init="random", n_init=4, random_state=42, normalized_stress="auto")
X_mds = mds.fit_transform(X_scaled)
print(f"MDS stress: {mds.stress_:.3f}")

# t-SNE
tsne = TSNE(n_components=2, perplexity=30, random_state=42, init="pca")
X_tsne = tsne.fit_transform(X_scaled)

# UMAP(インストール済みのときのみ)
try:
    import umap
    reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=None)
    X_umap = reducer.fit_transform(X_scaled)
    umap_available = True
except ImportError:
    print("umap-learn がインストールされていません。 `pip install umap-learn` で導入してください。")
    X_umap = None
    umap_available = False


### 手法比較

In [ ]:
# 4 手法の 2D 埋め込みを並べて比較(色=種の真ラベル)
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

embeddings = [
    (X_pca, f"PCA(累積寄与率 {pca.explained_variance_ratio_.sum():.1%})"),
    (X_mds, "MDS (PCoA, ユークリッド距離)"),
    (X_tsne, "t-SNE (perplexity=30)"),
    (X_umap, "UMAP (n_neighbors=15)"),
]
palette = {"Adelie": "#1b9e77", "Chinstrap": "#d95f02", "Gentoo": "#7570b3"}

for ax, (emb, title) in zip(axes.flat, embeddings):
    if emb is None:
        ax.text(0.5, 0.5, "UMAP 未インストール", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(title)
        ax.set_xticks([]); ax.set_yticks([])
        continue
    for sp, color in palette.items():
        mask = species_labels == sp
        ax.scatter(emb[mask, 0], emb[mask, 1], label=sp, color=color, alpha=0.7, s=30, edgecolor="white", linewidth=0.3)
    ax.set_title(title)
    ax.set_xlabel("Dim 1")
    ax.set_ylabel("Dim 2")
    ax.legend(fontsize=9)

plt.suptitle("4 つの次元圧縮手法による可視化(色=種の真ラベル)", y=1.00, fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# 同じ次元圧縮図を「真ラベル」と「クラスタリング結果」で色分けして比較
# (UMAP が使えなければ PCA を代わりに使用)
embedding_for_overlay = X_umap if umap_available else X_pca
embedding_name = "UMAP" if umap_available else "PCA"

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
label_sets = [
    ("species", "真ラベル(species)", "Set2"),
    ("cluster", "階層的クラスタリング", "Set1"),
    ("kmeans_cluster", "k-means", "Set1"),
    ("graph_cluster", "グラフクラスタリング", "Set1"),
]
for ax, (col, title, pal) in zip(axes, label_sets):
    sns.scatterplot(
        x=embedding_for_overlay[:, 0], y=embedding_for_overlay[:, 1],
        hue=df_clean[col].astype(str), palette=pal, ax=ax, s=30, alpha=0.8,
    )
    ax.set_title(title)
    ax.set_xlabel(f"{embedding_name} Dim 1")
    ax.set_ylabel(f"{embedding_name} Dim 2")
    ax.legend(fontsize=8, loc="best")

plt.suptitle(f"{embedding_name} 2D 埋め込み上で「真ラベル」と各クラスタリング結果を比較", y=1.02)
plt.tight_layout()
plt.show()


> **Q6：次元圧縮によって新たな発見はありましたか？**

## 10. じっくり考えよう

1. **手法の目的について**
   - GLM・ランダムフォレストは**教師あり**、階層的クラスタリング・K-means・グラフクラスタリングは**教師なし**でした。それぞれどのような時に有効な方法なのか、色々な適用例を想定しながら考えてみよう。

2. **「変数の効き」をどのように評価するか**
   - GLMの係数には符号(正/負)があるが、ランダムフォレストの重要度にはない。これはなぜでしょう？

3. **クラスタ数問題**
   - クラスタ数の決め方は各手法で異なりました。それぞれのPros/Consは？

4. **「距離」とは何か**
   - 線形手法(PCA / MDS)と非線形手法(t-SNE / UMAP)で「距離の意味」はどう違うでしょうか？

5. **データの質について**
   - 欠損値除去で約340→333サンプルに減りましたが、もし欠損が多いデータならどう対処するのが良いでしょうか？